# Vehicle Sensor Analytics & Predictive Maintenance
## Notebook 2: Anomaly Detection & Feature Engineering

**Approach:**  
- Rolling window statistics to capture signal drift over time  
- Per-subsystem Isolation Forest (unsupervised) — no labels needed  
- Anomaly scores validated against ground truth labels

**Output:** `df_anomaly.csv` — enriched dataframe for ML notebook

---
### Sections
1. Setup & Load
2. Rolling Window Features
3. Engineered Interaction Features
4. Isolation Forest — Engine Subsystem
5. Isolation Forest — Brake Subsystem
6. Isolation Forest — Battery Subsystem
7. Anomaly Score Analysis
8. Time-Series Overlay
9. Anomaly vs Ground Truth
10. Final Feature Summary & Save

---
## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

import os
os.makedirs('../outputs', exist_ok=True)

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('../data/df_eda.csv', parse_dates=['timestamp'])
df = df.sort_values(['vehicle_id', 'timestamp']).reset_index(drop=True)

targets = ['engine_failure_imminent', 'brake_issue_imminent', 'battery_issue_imminent']

print(f'Loaded: {df.shape}')
print(f'Vehicles: {df.vehicle_id.nunique()}')

---
## 2. Rolling Window Features

Computed **per vehicle** to avoid bleeding signals across different vehicles.  
Window = 5 records (~5 hours of data at hourly sampling).  
Captures gradual drift that a point-in-time reading misses.

In [ ]:
ROLLING_WINDOW = 5

# Signals worth tracking for drift
rolling_targets = {
    'engine': ['engine_temp_c', 'engine_rpm', 'oil_pressure_psi', 'coolant_temp_c', 'exhaust_gas_temp_c'],
    'brake':  ['brake_temp_c', 'brake_pad_wear_mm', 'brake_fluid_level_psi'],
    'battery': ['battery_voltage_v', 'battery_charge_percent', 'battery_health_percent', 'alternator_output_v']
}
all_rolling_signals = [s for signals in rolling_targets.values() for s in signals]

def add_rolling_features(group, cols, window=ROLLING_WINDOW):
    """Add rolling mean and std per vehicle group."""
    for col in cols:
        group[f'{col}_roll_mean'] = group[col].rolling(window, min_periods=1).mean()
        group[f'{col}_roll_std']  = group[col].rolling(window, min_periods=1).std().fillna(0)
    return group

df = df.groupby('vehicle_id', group_keys=False).apply(
    lambda g: add_rolling_features(g, all_rolling_signals)
)
df = df.reset_index(drop=True)

roll_cols = [c for c in df.columns if '_roll_' in c]
print(f'Rolling features added: {len(roll_cols)}')
print(roll_cols[:8], '...')

---
## 3. Engineered Interaction Features

Domain-informed features that encode physical relationships between sensors.

In [ ]:
# --- Engine ---
# Thermal stress: high temp + high RPM together is worse than either alone
df['thermal_stress'] = df['engine_temp_c'] * df['engine_load_percent'] / 100

# Temperature delta: engine vs coolant — large gap signals cooling system lag
df['temp_delta_engine_coolant'] = df['engine_temp_c'] - df['coolant_temp_c']

# Oil pressure efficiency: low pressure at high RPM = warning sign
df['oil_pressure_per_rpm'] = df['oil_pressure_psi'] / (df['engine_rpm'] + 1)

# --- Brake ---
# Wheel speed mismatch (already computed in EDA, re-derive cleanly)
wheel_cols = ['wheel_speed_fl_kph', 'wheel_speed_fr_kph', 'wheel_speed_rl_kph', 'wheel_speed_rr_kph']
df['wheel_speed_mean'] = df[wheel_cols].mean(axis=1)
df['wheel_speed_std']  = df[wheel_cols].std(axis=1)

# Speed vs brake pedal: pressing brake at high speed = brake stress event
df['brake_stress_event'] = df['vehicle_speed_kph'] * df['brake_pedal_pos_percent'] / 100

# --- Battery ---
# Alternator delta: if alternator output < battery voltage, charging is failing
df['alternator_deficit'] = df['battery_voltage_v'] - df['alternator_output_v']

# Charge efficiency proxy
df['charge_efficiency'] = df['battery_charge_percent'] * df['battery_health_percent'] / 100

engineered = ['thermal_stress', 'temp_delta_engine_coolant', 'oil_pressure_per_rpm',
              'wheel_speed_mean', 'wheel_speed_std', 'brake_stress_event',
              'alternator_deficit', 'charge_efficiency']

print('Engineered features:')
for f in engineered:
    print(f'  {f:35s}: mean={df[f].mean():.2f}, std={df[f].std():.2f}')

---
## 4. Isolation Forest — Engine Subsystem

Isolation Forest works by randomly partitioning features — anomalies require fewer splits to isolate.  
We train on **all data** (unsupervised) and compare scores against labels afterward.

In [ ]:
# Engine feature set: raw signals + rolling features + engineered
engine_features = [
    'engine_temp_c', 'engine_rpm', 'oil_pressure_psi', 'coolant_temp_c',
    'exhaust_gas_temp_c', 'engine_load_percent', 'fuel_consumption_lph',
    'engine_temp_c_roll_mean', 'engine_temp_c_roll_std',
    'engine_rpm_roll_mean', 'engine_rpm_roll_std',
    'oil_pressure_psi_roll_mean', 'oil_pressure_psi_roll_std',
    'thermal_stress', 'temp_delta_engine_coolant', 'oil_pressure_per_rpm'
]

# Contamination = approximate positive rate for engine failures
engine_contamination = df['engine_failure_imminent'].mean()
print(f'Engine contamination rate: {engine_contamination:.3f}')

scaler_engine = StandardScaler()
X_engine = scaler_engine.fit_transform(df[engine_features])

iso_engine = IsolationForest(
    n_estimators=200,
    contamination=engine_contamination,
    random_state=RANDOM_STATE
)
iso_engine.fit(X_engine)

# Raw anomaly score (more negative = more anomalous)
df['engine_anomaly_score'] = iso_engine.decision_function(X_engine)
# Binary flag: -1 → anomaly, 1 → normal → convert to 0/1
df['engine_anomaly_flag']  = (iso_engine.predict(X_engine) == -1).astype(int)

print(f'Engine anomalies detected: {df["engine_anomaly_flag"].sum()} ({df["engine_anomaly_flag"].mean()*100:.1f}%)')

---
## 5. Isolation Forest — Brake Subsystem

In [ ]:
brake_features = [
    'brake_fluid_level_psi', 'brake_pad_wear_mm', 'brake_temp_c',
    'abs_fault_indicator', 'brake_pedal_pos_percent',
    'brake_temp_c_roll_mean', 'brake_temp_c_roll_std',
    'brake_pad_wear_mm_roll_mean', 'brake_pad_wear_mm_roll_std',
    'wheel_speed_std', 'wheel_speed_mean', 'brake_stress_event'
]

brake_contamination = df['brake_issue_imminent'].mean()
print(f'Brake contamination rate: {brake_contamination:.3f}')

scaler_brake = StandardScaler()
X_brake = scaler_brake.fit_transform(df[brake_features])

iso_brake = IsolationForest(
    n_estimators=200,
    contamination=brake_contamination,
    random_state=RANDOM_STATE
)
iso_brake.fit(X_brake)

df['brake_anomaly_score'] = iso_brake.decision_function(X_brake)
df['brake_anomaly_flag']  = (iso_brake.predict(X_brake) == -1).astype(int)

print(f'Brake anomalies detected: {df["brake_anomaly_flag"].sum()} ({df["brake_anomaly_flag"].mean()*100:.1f}%)')

---
## 6. Isolation Forest — Battery Subsystem

In [ ]:
battery_features = [
    'battery_voltage_v', 'battery_current_a', 'battery_temp_c',
    'alternator_output_v', 'battery_charge_percent', 'battery_health_percent',
    'battery_voltage_v_roll_mean', 'battery_voltage_v_roll_std',
    'battery_charge_percent_roll_mean', 'battery_charge_percent_roll_std',
    'battery_health_percent_roll_mean', 'battery_health_percent_roll_std',
    'alternator_deficit', 'charge_efficiency'
]

battery_contamination = df['battery_issue_imminent'].mean()
print(f'Battery contamination rate: {battery_contamination:.3f}')

scaler_battery = StandardScaler()
X_battery = scaler_battery.fit_transform(df[battery_features])

iso_battery = IsolationForest(
    n_estimators=200,
    contamination=battery_contamination,
    random_state=RANDOM_STATE
)
iso_battery.fit(X_battery)

df['battery_anomaly_score'] = iso_battery.decision_function(X_battery)
df['battery_anomaly_flag']  = (iso_battery.predict(X_battery) == -1).astype(int)

print(f'Battery anomalies detected: {df["battery_anomaly_flag"].sum()} ({df["battery_anomaly_flag"].mean()*100:.1f}%)')

---
## 7. Anomaly Score Analysis

In [ ]:
# Score distribution: anomalous vs normal — should see clear separation
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

configs = [
    ('engine_anomaly_score', 'engine_failure_imminent', 'Engine Anomaly Score',   '#e74c3c'),
    ('brake_anomaly_score',  'brake_issue_imminent',   'Brake Anomaly Score',    '#e67e22'),
    ('battery_anomaly_score','battery_issue_imminent', 'Battery Anomaly Score',  '#2980b9'),
]

for ax, (score_col, label_col, title, color) in zip(axes, configs):
    normal  = df[df[label_col] == 0][score_col]
    failure = df[df[label_col] == 1][score_col]
    ax.hist(normal,  bins=40, alpha=0.6, color='#2ecc71', label='Normal',  density=True)
    ax.hist(failure, bins=40, alpha=0.6, color=color,     label='Failure', density=True)
    ax.axvline(0, color='black', linestyle='--', linewidth=1.2, label='Decision boundary')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Anomaly Score (lower = more anomalous)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Isolation Forest Anomaly Score Distributions\nvs Ground Truth Labels',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/11_anomaly_score_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: do anomaly flags co-occur with failure labels?
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (flag_col, label_col, title) in zip(axes, [
    ('engine_anomaly_flag',  'engine_failure_imminent', 'Engine'),
    ('brake_anomaly_flag',   'brake_issue_imminent',   'Brake'),
    ('battery_anomaly_flag', 'battery_issue_imminent', 'Battery'),
]):
    cm = confusion_matrix(df[label_col], df[flag_col])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Predicted Normal', 'Predicted Anomaly'],
                yticklabels=['Actual Normal', 'Actual Failure'])
    ax.set_title(f'{title} — Isolation Forest\nvs Ground Truth', fontweight='bold', fontsize=10)

plt.suptitle('Anomaly Flag vs Actual Failure Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/12_anomaly_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ROC-AUC for each subsystem — treats anomaly score as a ranking signal
print('Isolation Forest — ROC-AUC vs Ground Truth Labels')
print('(Note: IF is unsupervised — AUC > 0.5 means it has real signal)')
print('-' * 55)
for score_col, label_col, name in [
    ('engine_anomaly_score',  'engine_failure_imminent', 'Engine'),
    ('brake_anomaly_score',   'brake_issue_imminent',   'Brake'),
    ('battery_anomaly_score', 'battery_issue_imminent', 'Battery'),
]:
    # Negate score: IF scores are inverted (more negative = anomaly)
    auc = roc_auc_score(df[label_col], -df[score_col])
    print(f'  {name:10s}: AUC = {auc:.3f}')

---
## 8. Time-Series Overlay — Anomaly Flags on Sensor Data

In [ ]:
# Use a vehicle that actually has some failure events
# Find a vehicle with at least one failure across any target
vehicle_with_failure = (
    df[df[targets].any(axis=1)]
    .groupby('vehicle_id')
    .size()
    .idxmax()
)
vdf = df[df['vehicle_id'] == vehicle_with_failure].set_index('timestamp').sort_index()
print(f'Visualising vehicle: {vehicle_with_failure} | Brand: {vdf.brand.iloc[0]}')
print(f'Engine failures: {vdf["engine_failure_imminent"].sum()} | '
      f'Brake issues: {vdf["brake_issue_imminent"].sum()} | '
      f'Battery issues: {vdf["battery_issue_imminent"].sum()}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=True)

# --- ENGINE ---
ax = axes[0]
ax.plot(vdf.index, vdf['engine_temp_c'], color='#c0392b', linewidth=1, label='Engine Temp (°C)')
ax.plot(vdf.index, vdf['engine_temp_c_roll_mean'], color='#e74c3c',
        linewidth=1.5, linestyle='--', alpha=0.8, label='Rolling Mean (5-step)')
# Shade actual failure periods
ax.fill_between(vdf.index, vdf['engine_temp_c'].min(), vdf['engine_temp_c'].max(),
                where=vdf['engine_failure_imminent']==1,
                alpha=0.15, color='red', label='Actual Failure')
# Mark Isolation Forest anomalies
anomalies = vdf[vdf['engine_anomaly_flag'] == 1]
ax.scatter(anomalies.index, anomalies['engine_temp_c'],
           color='darkred', s=30, zorder=5, label='IF Anomaly', marker='x')
ax.set_ylabel('Engine Temp (°C)', fontsize=9)
ax.set_title(f'Vehicle {vehicle_with_failure} — Anomaly Detection Overlay', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')

# --- BRAKE ---
ax = axes[1]
ax.plot(vdf.index, vdf['brake_pad_wear_mm'], color='#d35400', linewidth=1, label='Brake Pad Wear (mm)')
ax.plot(vdf.index, vdf['brake_pad_wear_mm_roll_mean'], color='#e67e22',
        linewidth=1.5, linestyle='--', alpha=0.8, label='Rolling Mean')
ax.fill_between(vdf.index, vdf['brake_pad_wear_mm'].min(), vdf['brake_pad_wear_mm'].max(),
                where=vdf['brake_issue_imminent']==1,
                alpha=0.15, color='orange', label='Actual Issue')
brake_anomalies = vdf[vdf['brake_anomaly_flag'] == 1]
ax.scatter(brake_anomalies.index, brake_anomalies['brake_pad_wear_mm'],
           color='darkorange', s=30, zorder=5, label='IF Anomaly', marker='x')
ax.set_ylabel('Brake Pad Wear (mm)', fontsize=9)
ax.legend(fontsize=8, loc='upper right')

# --- BATTERY ---
ax = axes[2]
ax.plot(vdf.index, vdf['battery_voltage_v'], color='#1a5276', linewidth=1, label='Battery Voltage (V)')
ax.plot(vdf.index, vdf['battery_voltage_v_roll_mean'], color='#2980b9',
        linewidth=1.5, linestyle='--', alpha=0.8, label='Rolling Mean')
ax.fill_between(vdf.index, vdf['battery_voltage_v'].min(), vdf['battery_voltage_v'].max(),
                where=vdf['battery_issue_imminent']==1,
                alpha=0.15, color='blue', label='Actual Issue')
batt_anomalies = vdf[vdf['battery_anomaly_flag'] == 1]
ax.scatter(batt_anomalies.index, batt_anomalies['battery_voltage_v'],
           color='darkblue', s=30, zorder=5, label='IF Anomaly', marker='x')
ax.set_ylabel('Battery Voltage (V)', fontsize=9)
ax.set_xlabel('Timestamp', fontsize=9)
ax.legend(fontsize=8, loc='upper right')

plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../outputs/13_anomaly_timeseries_overlay.png', bbox_inches='tight')
plt.show()

---
## 9. Anomaly vs Ground Truth — Deeper Evaluation

In [ ]:
# Classification report for each subsystem
pairs = [
    ('engine_anomaly_flag',  'engine_failure_imminent', 'ENGINE'),
    ('brake_anomaly_flag',   'brake_issue_imminent',   'BRAKE'),
    ('battery_anomaly_flag', 'battery_issue_imminent', 'BATTERY'),
]

for flag_col, label_col, name in pairs:
    print(f'\n{name} — Isolation Forest vs Ground Truth')
    print('-' * 45)
    print(classification_report(
        df[label_col], df[flag_col],
        target_names=['Normal', 'Anomaly/Failure']
    ))

In [ ]:
# Anomaly score vs actual label — violin plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (score_col, label_col, name, color) in zip(axes, [
    ('engine_anomaly_score',  'engine_failure_imminent', 'Engine',  '#e74c3c'),
    ('brake_anomaly_score',   'brake_issue_imminent',   'Brake',   '#e67e22'),
    ('battery_anomaly_score', 'battery_issue_imminent', 'Battery', '#2980b9'),
]):
    plot_df = df[[score_col, label_col]].copy()
    plot_df[label_col] = plot_df[label_col].map({0: 'Normal', 1: 'Failure/Issue'})
    sns.violinplot(data=plot_df, x=label_col, y=score_col,
                   palette={'Normal': '#2ecc71', 'Failure/Issue': color},
                   inner='box', ax=ax)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'{name} Subsystem', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('IF Anomaly Score')

plt.suptitle('Anomaly Score Distribution\nby Actual Failure Label (violin plot)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/14_anomaly_violin_plots.png', bbox_inches='tight')
plt.show()

In [ ]:
# Co-occurrence: does flagging multiple subsystems simultaneously predict failure?
df['total_anomaly_flags'] = (df['engine_anomaly_flag'] +
                              df['brake_anomaly_flag'] +
                              df['battery_anomaly_flag'])
df['any_failure'] = df[targets].any(axis=1).astype(int)

cooccurrence = df.groupby('total_anomaly_flags')['any_failure'].mean().mul(100).round(1)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(cooccurrence.index, cooccurrence.values,
              color=['#2ecc71', '#f39c12', '#e67e22', '#e74c3c'],
              edgecolor='white', linewidth=1.2)
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['0 flags\n(all normal)', '1 subsystem\nanomaly',
                    '2 subsystems\nanomaly', '3 subsystems\nanomaly'])
ax.set_ylabel('% Records with Any Actual Failure')
ax.set_title('Multi-Subsystem Anomaly Co-occurrence\nvs Actual Failure Rate', fontweight='bold')
for bar, val in zip(bars, cooccurrence.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/15_multi_subsystem_cooccurrence.png', bbox_inches='tight')
plt.show()

---
## 10. Final Feature Summary & Save

In [ ]:
# All new columns added in this notebook
new_cols = roll_cols + engineered + [
    'engine_anomaly_score', 'engine_anomaly_flag',
    'brake_anomaly_score',  'brake_anomaly_flag',
    'battery_anomaly_score','battery_anomaly_flag',
    'total_anomaly_flags', 'any_failure'
]

print('=' * 60)
print('ANOMALY NOTEBOOK SUMMARY')
print('=' * 60)
print(f'\nTotal features in enriched dataset: {df.shape[1]}')
print(f'New features added this notebook:   {len(new_cols)}')
print(f'  Rolling features:    {len(roll_cols)}')
print(f'  Engineered features: {len(engineered)}')
print(f'  Anomaly scores:      3 (engine / brake / battery)')
print(f'  Anomaly flags:       3 (engine / brake / battery)')

print('\nKey insight:')
print('  Records with 2+ subsystem flags have significantly higher failure rates')
print('  → total_anomaly_flags is a strong composite feature for ML notebook')
print('\nNext: Notebook 03 — XGBoost + Random Forest + SHAP')
print('=' * 60)

In [ ]:
df.to_csv('../data/df_anomaly.csv', index=False)
print(f'Saved: ../data/df_anomaly.csv  |  Shape: {df.shape}')